In [1]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
os.makedirs('tables', exist_ok=True)

In [3]:
# Simulated

base_dir = 'experiments/synthetic_benchmark/results'
files = os.listdir(base_dir)
files = [f for f in files if f.endswith('.csv')]

merged = {}
cols = []
for f in files:
    id = f.replace('results_', '').replace('.csv', '')
    df = pd.read_csv(os.path.join(base_dir, f))
    df["model_config"] = pd.Categorical(df["model_config"], categories=['DextraDemixer', 'DextraDemixer+neg.', 'DextraDemixer+clone', 'DextraDemixer+neg.+clone', 'BEAM'], ordered=True)
    df = df.sort_values(["model_config", 'sim_config'])
    df = df.rename(columns={'sim_binding_ratio': 'fraction_of_binders', 'sim_mean_inc': 'signal_to_noise_ratio', 
                            'sim_nof_clones': 'num_clones', 'sim_p_binding_outlier': 'p_dropout', 'sim_rep': 'repetition_num', 
                            'sim_rng_key': 'sim_seed', 'sim_total_cells': 'total_cell_count', })
    

    if id != 'FDR':
        df = df.drop(columns=['cred_intvl', 'fdr', 'target_fdr'])
    if id == 'FDR':
        df = df.drop(columns=['f1', 'precision', 'aps', ])
        df = df[df['model_config'].isin(['DextraDemixer+neg.', 'DextraDemixer+neg.+clone'])]
        df = df[df['sim_num_binder'] > 1/df['target_fdr']]
    if id == 'scaling':
        df = df.drop(columns=['f1', 'precision', 'recall', 'aps', ])

    df = df.drop(columns=['Unnamed: 0', 'model_f_out', 'clone_median_p', 'tp', 'fp', 'tn',
       'fn', 'model_f', 'model_gex_key', 'model_airr_key', 'model_pmhc_key',
       'model_label_key', 'model_neg_ctrl_key', 'model_hc_scale',  'model_scaling_test',
       'modified_time', 'model_alpha_offset', 'model_seed', 'model_maxiter', 'model_hc_scale_prior',
       'model_lr_init_value', 'model_lr_end_value', 'model_lr_decay_rate',
       'model_lr_transition_steps', 'model_guide', 'posterior_config_clean',
       'sim_num_binder', 'source_fp',  'threshold',],
       errors="ignore",
    )
    order = ['config', 'model_config', 'sim_config', 'target_fdr', 'cred_intvl', 'fdr', 'f1', 'precision', 'recall', 'aps', 
    'cpu_model', 'total_time', 'peak_mem_resource', 'total_cell_count', 'fraction_of_binders', 'signal_to_noise_ratio', 
    'p_dropout', 'num_clones', 'repetition_num', 'sim_seed', 'model_f_in', ]

    df = df[[c for c in order if c in df.columns]]

    merged[id] = df

with pd.ExcelWriter(os.path.join('tables', 'SuppTable1.xlsx')) as writer:
    for id in ['benchmark', 'dropout', 'scaling', 'FDR']:
        merged[id].to_excel(writer, sheet_name=id, index=False)

In [4]:
# Gemuend

df = pd.read_csv('experiments/Gemuend_CMV/results/benchmark_results.csv', index_col=0)
df = df[['setting', 'model_config', 'f1', 'precision', 'recall', 'dex_key', 'sample', 'tcr_info_sequenced_only',]]
df = df.sort_values(['setting', 'model_config', ])
df.to_csv('tables/SuppTable2.csv', index=False)
df

,setting,model_config,f1,precision,recall,dex_key,sample,tcr_info_sequenced_only
39,DexA 5% (all cells),BEAM,0.913043,0.954545,0.875000,DexA,05-95-D2,False
38,DexA 5% (all cells),DextraDemixer,0.933333,1.000000,0.875000,DexA,05-95-D2,False
21,DexA 5% (has TCR),BEAM,1.000000,1.000000,1.000000,DexA,05-95-D2,True
20,DexA 5% (has TCR),DextraDemixer,1.000000,1.000000,1.000000,DexA,05-95-D2,True
22,DexA 5% (has TCR),ICON,0.823529,0.700000,1.000000,DexA,05-95-D2,True
24,DexA 5% (has TCR),ICON-code,0.000000,0.000000,0.000000,DexA,05-95-D2,True
23,DexA 5% (has TCR),ITRAP,0.245614,0.140000,1.000000,DexA,05-95-D2,True
35,DexA 50% (all cells),BEAM,0.928105,0.972603,0.887500,DexA,50-50-D2,False
34,DexA 50% (all cells),DextraDemixer,0.941935,0.973333,0.912500,DexA,50-50-D2,False
11,DexA 50% (has TCR),BEAM,1.000000,1.000000,1.000000,DexA,50-50-D2,True


In [5]:
# Kocher

df1 = pd.read_csv('experiments/Kocher_SARS-CoV-2/results/kocher/classification_metrics.csv', index_col=0)
df2 = pd.read_csv('experiments/Kocher_SARS-CoV-2/results/kocher/separation_metrics.csv', index_col=0)

df = pd.concat([df1, df2], ignore_index=True)
df = df[df['Baseline'] == 'paper']
df = df[df['Metric'].isin(['F1-Score', 'Precision', 'Recall', 'APS'])]
df['Method'] = df['Method'].replace({'Base': 'DextraDemixer', 'CloneModel': 'DextraDemixer+clone'})
df['Setting'] = df['Epitope'] + '_' + df['Experiment']
df = df.drop(columns='Baseline')
df = df.pivot_table(
    index=["Method", "Epitope", "Experiment", "Setting"],
    columns="Metric",
    values="Value",
).reset_index()
df.columns.name = None
df.to_csv('tables/SuppTable3.csv', index=False)
df

,Method,Epitope,Experiment,Setting,APS,F1-Score,Precision,Recall
0,DextraDemixer,LTD,first,LTD_first,1.000000,1.000000,1.000000,1.000000
1,DextraDemixer,LTD,second,LTD_second,0.961698,0.916031,1.000000,0.845070
2,DextraDemixer,LTD,third,LTD_third,0.956786,0.808989,1.000000,0.679245
3,DextraDemixer,YLQ,first,YLQ_first,1.000000,1.000000,1.000000,1.000000
4,DextraDemixer,YLQ,second,YLQ_second,0.928802,0.887218,1.000000,0.797297
5,DextraDemixer,YLQ,third,YLQ_third,0.986111,0.941176,0.888889,1.000000
6,DextraDemixer+clone,LTD,first,LTD_first,1.000000,1.000000,1.000000,1.000000
7,DextraDemixer+clone,LTD,second,LTD_second,0.976646,0.958435,1.000000,0.920188
8,DextraDemixer+clone,LTD,third,LTD_third,0.983720,0.868327,1.000000,0.767296
9,DextraDemixer+clone,YLQ,first,YLQ_first,1.000000,1.000000,1.000000,1.000000
